In [ ]:
# Before committing to the architecture
print(train_set_sanitized.count(), "rows")
# Rough estimate: pull one group, extrapolate
sample = train_set_sanitized.filter(
    F.col("PARENT_DEALER_CODE_MODEL_FAMILY") == group_keys[0]
).to_pandas()
print(f"One series: {sample.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Estimated total: {sample.memory_usage(deep=True).sum() / 1e6 * len(group_keys) / 1e3:.1f} GB")

In [ ]:
# Check one chunk's memory footprint
sample_chunk = train_set_sanitized.filter(
    F.col("PARENT_DEALER_CODE").isin(dealer_chunks[0])
).to_pandas()

print(f"One chunk RAM: {sample_chunk.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print(f"Estimated all 4 chunks: {sample_chunk.memory_usage(deep=True).sum() / 1e9 * 4:.1f} GB")

### Chunking by dealers

In [ ]:
dealer_keys = [
    row["PARENT_DEALER_CODE"] 
    for row in train_set_sanitized.select("PARENT_DEALER_CODE").distinct().collect()
]

chunk_size = 300
dealer_chunks = [dealer_keys[i:i+chunk_size] for i in range(0, len(dealer_keys), chunk_size)]
print(f"{len(dealer_chunks)} chunks of up to {chunk_size} dealers each")

In [ ]:
for idx, chunk in enumerate(dealer_chunks):
    chunk_df = train_set_sanitized.filter(F.col("PARENT_DEALER_CODE").isin(chunk))
    chunk_path = f"{stage_train_path}/chunk_{idx:04d}.parquet"
    chunk_df.write.parquet(chunk_path)
    print(f"Written chunk {idx+1}/{len(dealer_chunks)}")

In [ ]:
get_from_stage(session, stage_train_path, local_train_dir)

In [ ]:
all_target_series = []
all_future_cov_series = []
all_static_rows = []

for chunk_file in sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet"))):
    df = pd.read_parquet(chunk_file)
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values([group_col, time_col])
    
    for key, group_df in df.groupby(group_col):
        group_df = group_df.reset_index(drop=True)
        
        ts_target = TimeSeries.from_dataframe(
            group_df, time_col=time_col, value_cols=target_col, freq='D'
        )
        ts_future = TimeSeries.from_dataframe(
            group_df, time_col=time_col, value_cols=future_covariates, freq='D'
        )
        static_row = group_df[static_covariates].iloc[0].to_dict()
        static_row[group_col] = key
        
        all_target_series.append(ts_target)
        all_future_cov_series.append(ts_future)
        all_static_rows.append(static_row)

static_df = pd.DataFrame(all_static_rows)